In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:

# Load the training and testing datasets
train_data = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

test_data = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

# Convert labels into numerical values
train_data['label'] = train_data['label'].map({
    'ham': 0,
    'spam': 1
})

test_data['label'] = test_data['label'].map({
    'ham': 0,
    'spam': 1
})

# Separate messages and labels
train_messages = train_data['message'].astype(str).values
train_labels = train_data['label'].values

test_messages = test_data['message'].astype(str).values
test_labels = test_data['label'].values

max_tokens = 10000

vectorize_layer = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='tf_idf',
    ngrams=2,
    dtype=tf.string
)

vectorize_layer.adapt(train_messages)

print("Training samples:", len(train_messages))
print("Testing samples:", len(test_messages))

print("\nFirst training examples:")
print(train_data.head())


In [ ]:

# Build the neural network
model = keras.Sequential([
    vectorize_layer,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    train_messages,
    train_labels,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate the model
loss, accuracy = model.evaluate(
    test_messages,
    test_labels,
    verbose=2
)

print("Test accuracy:", accuracy)


In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):

    # Convert the message into a TensorFlow string tensor
    input_text = tf.constant([pred_text], dtype=tf.string)

    # Predict the probability of spam
    prediction = model.predict(
        input_text,
        verbose=0
    )[0][0]

    # Determine the message category
    label = 'spam' if prediction >= 0.5 else 'ham'

    return [float(prediction), label]


pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
